In [1]:

import requests
from bs4 import BeautifulSoup


In [2]:
# Run the scraper, which saves the rendered homepage to lsm_page.html
!python scrape.py
html = open("lsm_page.html", encoding="utf-8").read()
doc = BeautifulSoup(html, "html.parser")

saved lsm_page.html


In [3]:
articles = []
for item in doc.select("article.list-article"):
    a = item.select_one("a")
    if not a:
        continue
    article = {
        'url': "https://eng.lsm.lv" + a['href'],
        'headline': a.get_text(strip=True),
    }
    articles.append(article)
len(articles)

56

In [4]:
import pandas as pd

df = pd.DataFrame(articles)
df.head()

,url,headline
0,https://eng.lsm.lv/article/weather/weather/12....,Storm warning for eastern Latvia on Sunday eve...
1,https://eng.lsm.lv/article/economy/transport/1...,Death on railway line in Rīga
2,https://eng.lsm.lv/article/features/features/1...,The Deep Roots of Resilience: what a crown tau...
3,https://eng.lsm.lv/article/economy/employment/...,4.8% of Latvia's jobs are in culture
4,https://eng.lsm.lv/article/society/society/12....,Turkish citizens killed in Latvian car crash


In [5]:
import os

# Try to create a folder called 'data'
# and if it exists DON'T THROW AN ERROR
os.makedirs("data\lsm_lv", exist_ok=True)

<>:5: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<>:5: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
C:\Users\jpjul\AppData\Local\Temp\ipykernel_25784\2556681107.py:5: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
  os.makedirs("data\lsm_lv", exist_ok=True)


In [6]:
from datetime import datetime

# This would keep track down to the second
datetime.now().strftime("%Y-%m-%d_%H.%M.%S")

# This only does the day
date_string = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")
filepath = f"data/lsm_lv/{date_string}.csv"

df.to_csv(filepath, index=False)

In [7]:
# Append to an always-updated file, deduplicated by url.

# add columns for when we scraped this
df['scrape_date'] = datetime.now().strftime("%Y-%m-%d")
df['scrape_datetime'] = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")

# open the always-updated file if it exists, else start blank
try:
    existing_df = pd.read_csv("lsm_lv-always-updated.csv")
except:
    existing_df = pd.DataFrame([])

# OLD first, then NEW, so keep='first' preserves the earliest scrape date
combined = pd.concat([existing_df, df], ignore_index=True)
print("Before dropping duplicates:", len(combined))

# dedup on url ONLY -> only genuinely new articles get added to the file
combined = combined.drop_duplicates(subset=['url'], keep='first')
print("After dropping duplicates: ", len(combined))

combined.to_csv("lsm_lv-always-updated.csv", index=False)
combined.head()

Before dropping duplicates: 56
After dropping duplicates:  48


,url,headline,scrape_date,scrape_datetime
0,https://eng.lsm.lv/article/weather/weather/12....,Storm warning for eastern Latvia on Sunday eve...,2026-07-12,2026-07-12_21.45.45
1,https://eng.lsm.lv/article/economy/transport/1...,Death on railway line in Rīga,2026-07-12,2026-07-12_21.45.45
2,https://eng.lsm.lv/article/features/features/1...,The Deep Roots of Resilience: what a crown tau...,2026-07-12,2026-07-12_21.45.45
3,https://eng.lsm.lv/article/economy/employment/...,4.8% of Latvia's jobs are in culture,2026-07-12,2026-07-12_21.45.45
4,https://eng.lsm.lv/article/society/society/12....,Turkish citizens killed in Latvian car crash,2026-07-12,2026-07-12_21.45.45
